# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the available record sets and their fields, referencing all entities by their `@id`.

In [ ]:
# Overview: List available record sets and fields by `@id`

# Retrieve record sets
record_sets = dataset.record_sets
print("Available record sets and fields:")

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}  name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict): # single field as dict
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', None)
        field_name = field.get('name', '<no name>')
        print(f"    -> Field @id: {field_id}  name: {field_name}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Here, we use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set, referencing by @id

import warnings
warnings.filterwarnings("ignore")

dataframes = {}

# Collect all record set @ids for iteration
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Display columns for one record set
if record_set_ids:
    show_id = record_set_ids[0]
    print(f"Columns for record set {show_id}:")
    print(dataframes[show_id].columns.tolist())
    print(dataframes[show_id].head())

## 4. Exploratory Data Analysis (EDA)
We apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by attributes.

Remember: All fields and columns are referenced by their `@id`.

In [ ]:
# Select a record set for analysis
# For demonstration, use the first record set id

record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# If there are any numeric fields, use one for EDA
numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical column if available
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA. Please review the dataset structure.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below, we plot the distribution of the selected numeric field (if available) and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    numeric_field_id = numeric_fields[0]
    filtered_df = df[df[numeric_field_id] > 10]
    if not filtered_df.empty:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
        plt.title(f'Distribution of {numeric_field_id} in Record Set {record_set_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
        
        if f"{numeric_field_id}_normalized" in filtered_df.columns:
            plt.figure(figsize=(8,4))
            sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=10, kde=True)
            plt.title(f'Normalized Distribution of {numeric_field_id} in Record Set {record_set_id}')
            plt.xlabel(f"{numeric_field_id}_normalized")
            plt.ylabel('Count')
            plt.show()
    else:
        print('No records above threshold for plotting.')
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we explored the FAIR^2 clinical dataset using Croissant schema-driven loading and referenced all entities by their `@id`. Key steps included:

- Loading metadata and records using the Croissant URL.
- Reviewing available record sets, fields, and columns by their `@id`.
- Extracting each record set into pandas DataFrames.
- Performing basic exploratory analysis and normalizations on available numeric fields.
- Visualizing distributions of key numeric attributes.

This small dataset offers structured clinical, genetic, and imaging features for rare infertility management cases, but due to sample limitations, its use is best suited for case illustration and academic study. All processing steps were performed in alignment with the Croissant FAIR data principles.